# Episodic Memory를 사용하는 Debugging Assistant

## 개요

이 Notebook에서는 reflection이 포함된 **AgentCore Episodic Memory**를 사용하여 지능형 **Debugging Assistant**를 구축하는 방법을 살펴봅니다. Agent는 이전 debugging session에서 학습하고 과거 경험을 바탕으로 맥락에 맞는 지침을 제공합니다.

### Episodic Memory란?

**Episodic Memory**는 구조화된 맥락과 함께 전체 상호 작용 순서(episode)를 포착합니다. 개별 fact를 저장하는 semantic memory와 달리 episodic memory는 다음 정보를 보존합니다.
- **전체 대화 흐름**: 문제 설명부터 해결까지의 전체 debugging session
- **시간적 맥락**: 수행한 작업의 순서와 시점
- **결과**: Debugging 시도의 성공 또는 실패 여부
- **구조화된 turn**: 생각, 작업, 관찰을 포함한 개별 단계

![Episodic memory](./episodic_memory.png)

### Reflection이란?

**Reflections**는 여러 episode에서 자동으로 추출하고 종합한 insight입니다. 다음 정보를 제공합니다.
- **Pattern 인식**: 유사한 episode 전반의 공통 문제와 해결책
- **Best practice**: 성공적인 debugging session에서 효과가 있었던 strategy
- **일반적인 함정**: 실패한 시도를 바탕으로 피해야 할 실수
- **전략적 지침**: 유사한 문제에 접근하기 위한 상위 수준의 조언

**출력 구조:**
- **Episodes**: `debugging/{actorId}/sessions/{sessionId}`에 저장되는 전체 대화 trace
- **Reflections**: `debugging/{actorId}`에 저장되는 여러 episode의 종합 지식

### Episodic Memory를 사용해야 하는 경우

다음과 같은 경우 episodic memory를 사용합니다.
1. **순차적 맥락이 중요한 경우**: 작업 순서와 결과가 중요함(예: debugging workflow, troubleshooting 절차)
2. **경험을 통한 학습이 필요한 경우**: Agent가 과거의 성공과 실패를 분석하여 개선되어야 함
3. **Process 검색이 필요한 경우**: 사용자가 "how did I solve X last time?" 또는 "show me the exact steps taken"을 기억해야 함

### 튜토리얼 세부 정보

| 항목 | 세부 정보 |
|:------------|:--------|
| 튜토리얼 유형 | Reflection이 포함된 Episodic Memory |
| Agent 유형 | Debugging Assistant |
| Framework | Strands Agents |
| LLM model | Claude Haiku 4.5 |
| Memory strategy | Reflection Configuration이 포함된 Episodic Memory |
| 난이도 | 중급 |

## 사전 요구 사항

- Python 3.10+
- AgentCore Memory 권한이 있는 AWS credentials
- AgentCore service에 대한 액세스

## 1단계: Dependency 설치 및 설정

In [ ]:
%pip install -qr requirements.txt

In [ ]:
import json
import logging
import uuid
from datetime import datetime, timezone
from typing import List, Dict
from pprint import pprint

# Logging 설정
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("debugging-assistant")

# Control plane 및 data plane 작업을 위해 boto3 import
import boto3

# Strands Agent framework 가져오기
from strands import Agent, tool

logger.info("✅ All dependencies imported successfully")

In [ ]:
import os

# 구성
REGION = os.getenv("AWS_REGION", "us-west-2")
# Session 식별자
ACTOR_ID = "developer"

logger.info(f"Configuration set for region: {REGION}")
logger.info(f"Actor ID: {ACTOR_ID}")

## 2단계: Episodic Strategy로 Memory 생성

**Reflection Configuration**이 포함된 **Episodic Memory Strategy**로 memory resource를 구성합니다. 이를 통해 다음 기능을 사용할 수 있습니다.
- 전체 debugging session episode 저장
- 여러 episode에서 reflection insight 자동 생성

In [ ]:
# Control plane 및 data plane 작업용 boto3 client 초기화
client = boto3.client(
    "bedrock-agentcore",
    region_name=REGION,
)
memory_client = boto3.client(
    "bedrock-agentcore-control",
    region_name=REGION,
)

In [ ]:
# Reflection을 포함한 episodic memory strategy를 dictionary로 정의
memory_name = "DebugAssistantEpisodic"

# Episodic memory는 customMemoryStrategy로 구현됨
episodic_strategy = {
    "episodicMemoryStrategy": {
        "name": "DebuggingEpisodeExtractor",
        "description": "Creates debugging session episodes with reflections per actor",
        "namespaceTemplates": ["debugging/{actorId}/sessions/{sessionId}/"],
        "reflectionConfiguration": {
            "namespaceTemplates": [
                "debugging/{actorId}/"  # Episodic memory namespace의 정확한 prefix여야 함
            ]
        },
    }
}
logger.info(f"Strategy configured: {episodic_strategy['episodicMemoryStrategy']['name']}")
logger.info(f"Episode namespace: {episodic_strategy['episodicMemoryStrategy']['namespaceTemplates'][0]}")
logger.info(
    f"Reflection namespace: {episodic_strategy['episodicMemoryStrategy']['reflectionConfiguration']['namespaceTemplates'][0]}"
)

In [ ]:
# Memory 조회 또는 생성
try:
    # 먼저 기존 memory 검색 시도
    list_response = memory_client.list_memories(maxResults=100)
    memory_id = None
    for mem in list_response.get("memories", []):
        detail = memory_client.get_memory(memoryId=mem["id"])
        if detail["memory"].get("name") == memory_name:
            memory_id = mem["id"]
            logger.info(f"✅ Using existing memory: {memory_id}")
            break

    # 찾지 못하면 생성
    if not memory_id:
        logger.info(f"Creating new memory: {memory_name}")
        response = memory_client.create_memory(
            name=memory_name,
            description="Episodic memory for debugging assistant with reflections",
            eventExpiryDuration=90,
            memoryStrategies=[episodic_strategy],
            clientToken=str(uuid.uuid4()),
        )
        memory_id = response["memory"]["id"]
        logger.info(f"✅ Memory created: {memory_id}")

        # ACTIVE 상태까지 대기
        import time

        for _ in range(30):
            status = memory_client.get_memory(memoryId=memory_id)["memory"]["status"]
            if status == "ACTIVE":
                logger.info("✅ Memory is ACTIVE")
                break
            time.sleep(10)

except Exception as e:
    logger.error(f"❌ Failed to get/create memory: {e}")
    raise

## 4단계: 이전 Debugging Session으로 Memory 채우기

이전 debugging session을 episodic memory에 불러오겠습니다. 각 session은 전체 debugging workflow를 나타냅니다.

In [ ]:
import os
import glob

# 모든 session data 파일 불러오기
data_dir = "./data"
session_files = sorted(glob.glob(f"{data_dir}/*.json"))

logger.info(f"Found {len(session_files)} session files to hydrate")

# 각 session 채우기
for session_file in session_files:
    session_name = os.path.basename(session_file).replace(".json", "")
    session_id = f"{session_name}_{datetime.now().strftime('%Y%m%d%H%M%S')}"

    logger.info(f"Hydrating session: {session_name}")

    # 대화 data 불러오기
    with open(session_file, "r") as f:
        conversation = json.load(f)

    # Payload 형식으로 변환
    payload = []
    for turn in conversation:
        conv_data = turn["conversational"]
        payload.append(
            {
                "conversational": {
                    "content": {"text": conv_data["content"]["text"]},
                    "role": conv_data["role"],
                }
            }
        )

    # Boto3를 직접 사용하여 event 생성
    event_timestamp = datetime.now(timezone.utc)
    result = client.create_event(
        memoryId=memory_id,
        actorId=ACTOR_ID,
        sessionId=session_id,
        eventTimestamp=event_timestamp,
        payload=payload,
    )

    logger.info(f"   ✓ Stored {len(payload)} turns - Event ID: {result['event']['eventId']}")

logger.info(f"✅ Successfully hydrated {len(session_files)} debugging sessions")

In [ ]:
### 장기 메모리로 추출되었는지 확인하기 위해 memory record 나열
import time
import pprint

reflection_namespace = f"debugging/{ACTOR_ID}/"
# time.sleep(60)
# Boto3 client를 직접 사용하여 memory record 검색
response = client.list_memory_records(memoryId=memory_id, namespace=reflection_namespace, maxResults=20)
memories = response.get("memoryRecordSummaries", [])
logger.info(f"   Found {len(memories)} memories")
if memories:
    pprint.pp(json.loads(memories[0]["content"]["text"]))

In [ ]:
# Reflection과 episode의 생성 여부 확인
import pprint

# Boto3 client를 직접 사용하여 memory record 검색
response = client.retrieve_memory_records(
    memoryId=memory_id,
    namespace=f"debugging/{ACTOR_ID}/",
    searchCriteria={
        "searchQuery": "memory leaks",
        "metadataFilters": [
            {
                "left": {"metadataKey": "x-amz-agentcore-memory-recordType"},
                "operator": "EQUALS_TO",
                "right": {"metadataValue": {"stringValue": "REFLECTION"}},
            }
        ],
        "topK": 10,
    },
    maxResults=20,
)

reflections = response.get("memoryRecordSummaries", [])
logger.info(f"   Found {len(reflections)} relevant reflections")
if reflections:
    for reflection in reflections:
        reflection_json = json.loads(reflection["content"]["text"])
        pprint.pp(reflection_json)

## 5단계: Memory 검색 Tool 생성

Agent를 위한 두 가지 전문 tool을 만듭니다.
1. **retrieve_process**: 세부적인 단계별 process를 위해 전체 episode trace 검색
2. **retrieve_reflection_knowledge**: 여러 episode에서 종합한 insight와 pattern 검색

In [ ]:
def count_tokens(text: str) -> int:
    """텍스트 문자열의 토큰 수를 근사합니다."""
    return len(text)


def linearize_episodes(episodes: List[Dict], include_steps: bool = True, include_reflections: bool = True) -> str:
    """에피소드 데이터를 사람이 읽기 쉬운 형식으로 선형화합니다."""
    if not episodes:
        return "No relevant episodes found."

    output = []
    for idx, episode in enumerate(episodes, 1):
        content = episode.get("content", {})

        # Text field에서 JSON parsing
        if "text" in content:
            try:
                episode_data = json.loads(content["text"])
            except json.JSONDecodeError:
                output.append(f"Episode {idx}: Unable to parse content\n")
                continue
        else:
            output.append(f"Episode {idx}: No content available\n")
            continue

        output.append(f"{'=' * 80}\nEpisode {idx}\n{'=' * 80}")
        output.append(f"**Situation:** {episode_data.get('situation', 'N/A')}")
        output.append(f"**Intent:** {episode_data.get('intent', 'N/A')}")
        output.append(f"**Assessment:** {episode_data.get('assessment', 'N/A')}\n")

        if include_steps:
            turns = episode_data.get("turns", [])
            if turns:
                output.append("**Debugging Steps:**")
                for turn_idx, turn in enumerate(turns, 1):
                    output.append(f"\nStep {turn_idx}:")
                    output.append(f"  Situation: {turn.get('situation', 'N/A')}")
                    output.append(f"  Action: {turn.get('action', 'N/A')}")
                    output.append(f"  Thought: {turn.get('thought', 'N/A')}")

        if include_reflections:
            reflection = episode_data.get("reflection")
            if reflection:
                output.append(f"\n**Reflection:** {reflection}\n")

    result = "\n".join(output)
    logger.info(f"   Episode tokens: {count_tokens(result)}")
    return result


def linearize_reflections(reflections: List[Dict]) -> str:
    """성찰 지식을 사람이 읽기 쉬운 형식으로 선형화합니다."""
    if not reflections:
        return "No reflection knowledge found."

    output = []
    for idx, reflection in enumerate(reflections, 1):
        content = reflection.get("content", {})
        score = reflection.get("score", 0)

        # Text field에서 JSON parsing
        if "text" in content:
            try:
                reflection_data = json.loads(content["text"])
            except json.JSONDecodeError:
                continue
        else:
            continue

        output.append(f"{'=' * 80}\nReflection {idx} (Relevance: {score:.2f})\n{'=' * 80}")
        output.append(f"**Title:** {reflection_data.get('title', 'Untitled')}")
        output.append(f"**Use Cases:** {reflection_data.get('use_cases', 'N/A')}")
        output.append(f"**Hints:** {reflection_data.get('hints', 'N/A')}\n")

    result = "\n".join(output)
    logger.info(f"   Reflection tokens: {count_tokens(result)}")
    return result


logger.info("✅ Linearization helper functions created")

In [ ]:
# Agent용 memory 검색 tool 생성


@tool
def retrieve_process(task: str, include_steps: bool = True) -> str:
    """
    Retrieve example processes to help solve the given task. Returns complete debugging
    episodes with configurable detail level.

    Use include_steps parameter to control verbosity:
    - Set include_steps=True when user asks for "exact steps", "full details", "how did we",
      "what steps did we take", or needs complete procedural information
    - Set include_steps=False for pattern/best practice queries where high-level context
      (situation, intent, assessment) is sufficient without step-by-step details

    Args:
        task: The task to solve that requires example processes
        include_steps: Whether to include detailed step-by-step turns (default: True)

    Returns:
        Formatted debugging episodes with optional detailed steps
    """
    logger.info(f"🔍 Retrieving processes for task: {task} (include_steps={include_steps})")

    try:
        # Episode namespace에서 검색
        namespace = f"debugging/{ACTOR_ID}/sessions/{session_id}/"

        # Boto3 client를 직접 사용하여 memory record 검색
        response = client.retrieve_memory_records(
            memoryId=memory_id,
            namespace=namespace,
            searchCriteria={"searchQuery": task, "topK": 3},
            maxResults=20,
        )

        episodes = response.get("memoryRecordSummaries", [])
        logger.info(f"   Found {len(episodes)} relevant episodes")

        # 구성 가능한 세부 수준으로 linearize
        return linearize_episodes(episodes, include_steps=include_steps, include_reflections=True)

    except Exception as e:
        logger.error(f"Error retrieving processes: {e}")
        return f"Error retrieving processes: {str(e)}"


@tool
def retrieve_reflection_knowledge(task: str, k: int = 5) -> str:
    """
    Retrieve synthesized reflection knowledge from past agent experiences. Each knowledge
    entry contains: (1) a descriptive title, (2) specific use cases for when to apply it,
    and (3) actionable hints including best practices from successful episodes and common
    pitfalls to avoid from failed episodes. Use this to get strategic guidance and patterns
    for similar tasks.

    Args:
        task: The current task to get strategic guidance for
        k: Number of reflection entries to retrieve (default: 5)

    Returns:
        Synthesized reflection knowledge from past debugging experiences
    """
    logger.info(f"🔍 Retrieving reflection knowledge for task: {task}")

    try:
        # Reflection namespace에서 검색
        namespace = f"debugging/{ACTOR_ID}/"

        # Boto3 client를 직접 사용하여 memory record 검색
        response = client.retrieve_memory_records(
            memoryId=memory_id,
            namespace=namespace,
            searchCriteria={
                "searchQuery": "memory leaks",
                "metadataFilters": [
                    {
                        "left": {"metadataKey": "x-amz-agentcore-memory-recordType"},
                        "operator": "EQUALS_TO",
                        "right": {"metadataValue": {"stringValue": "REFLECTION"}},
                    }
                ],
                "topK": k,
            },
            maxResults=20,
        )

        reflections = response.get("memoryRecordSummaries", [])
        logger.info(f"   Found {len(reflections)} relevant reflection insights")

        # reflection linearize 수행
        return linearize_reflections(reflections)

    except Exception as e:
        logger.error(f"Error retrieving reflections: {e}")
        return f"Error retrieving reflections: {str(e)}"


logger.info("✅ Memory retrieval tools created")

## 6단계: Debugging Assistant Agent 생성

이제 memory 검색 tool을 갖춘 Strands agent를 만듭니다.

In [ ]:
# Debugging assistant agent 생성
debugging_agent = Agent(
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[retrieve_process, retrieve_reflection_knowledge],
    system_prompt="""You are an expert Debugging Assistant with access to episodic memory.

Your capabilities:
- Retrieve past debugging episodes with complete step-by-step processes
- Access synthesized reflection knowledge showing patterns and best practices
- Provide guidance based on successful debugging experiences
- Warn about common pitfalls observed in past failures

When helping users:
1. Use retrieve_reflection_knowledge for strategic guidance, patterns, and high-level advice
2. Use retrieve_process when users need exact steps or want to recall what was done in a specific session
3. Synthesize insights from memory with your own reasoning
4. Be specific and actionable in your recommendations

Always explain your reasoning and cite relevant past experiences when available.""",
)

logger.info("✅ Debugging assistant agent created")

## 7단계: Debugging Assistant 테스트

다양한 시나리오를 테스트하여 agent가 episodic memory와 reflection을 어떻게 사용하는지 확인해 보겠습니다.

### 테스트 1: 전략적 지침 질의 (Reflection Knowledge)

In [ ]:
# 테스트 1: Memory 문제에 대한 전략적 지침 조회
query1 = "My application is running out of memory when processing large datasets. What should I look for?"

logger.info(f"\n{'=' * 80}")
logger.info("Test 1: Memory Issue Guidance")
logger.info(f"{'=' * 80}")
logger.info(f"Query: {query1}\n")

response1 = debugging_agent(query1)

print("\n" + "=" * 80)
print("AGENT RESPONSE:")
print("=" * 80)
print(response1)

### 테스트 2: 구체적인 Process 세부 정보 질의

In [ ]:
# 테스트 2: 구체적인 debugging process 조회
query2 = "Show me the exact steps for debugging a timeout issue with external API calls."

logger.info(f"\n{'=' * 80}")
logger.info("Test 2: API Timeout Debugging Process")
logger.info(f"{'=' * 80}")
logger.info(f"Query: {query2}\n")

response2 = debugging_agent(query2)

print("\n" + "=" * 80)
print("AGENT RESPONSE:")
print("=" * 80)
print(response2)

### 테스트 3: Pattern 인식 질의

In [ ]:
# 테스트 3: 동시성 문제의 pattern 및 best practice 조회
query3 = "What are common patterns and best practices for handling race conditions in multi-threaded applications?"

logger.info(f"\n{'=' * 80}")
logger.info("Test 3: Race Condition Patterns")
logger.info(f"{'=' * 80}")
logger.info(f"Query: {query3}\n")

response3 = debugging_agent(query3)

print("\n" + "=" * 80)
print("AGENT RESPONSE:")
print("=" * 80)
print(response3)

### 테스트 4: 특정 Session 회상

In [ ]:
# 테스트 4: Memory leak session에서 수행한 작업 회상
query4 = "What debugging steps did we take when we encountered the memory leak issue? I need the full details."

logger.info(f"\n{'=' * 80}")
logger.info("Test 4: Recall Memory Leak Session")
logger.info(f"{'=' * 80}")
logger.info(f"Query: {query4}\n")

response4 = debugging_agent(query4)

print("\n" + "=" * 80)
print("AGENT RESPONSE:")
print("=" * 80)
print(response4)

## 8단계: Memory 직접 검사

Episodic memory와 reflection에 저장된 내용을 직접 살펴보겠습니다.

In [ ]:
# Boto3를 사용하여 episode 직접 검사
logger.info("" + "=" * 80)
logger.info("Direct Episode Inspection")
logger.info("=" * 80)

# Boto3를 직접 사용하여 episode 검색
# NamespacePath를 사용하여 모든 session의 episode 검색 (계층적 일치)
namespace_path = f"debugging/{ACTOR_ID}/sessions/"
response = client.retrieve_memory_records(
    memoryId=memory_id,
    namespacePath=namespace_path,
    searchCriteria={"searchQuery": "debugging", "topK": 2},
    maxResults=10,
)

episodes = response.get("memoryRecordSummaries", [])

print(f"Found {len(episodes)} episodes in memory:")
for idx, episode in enumerate(episodes, 1):
    print(f"Episode {idx}:")
    pprint.pp(episode, depth=2, width=100)
    print("-" * 80)

In [ ]:
import pprint

response = client.retrieve_memory_records(
    memoryId=memory_id,
    namespace=reflection_namespace,
    searchCriteria={
        "searchQuery": "memory leaks",
        "metadataFilters": [
            {
                "left": {"metadataKey": "x-amz-agentcore-memory-recordType"},
                "operator": "EQUALS_TO",
                "right": {"metadataValue": {"stringValue": "REFLECTION"}},
            }
        ],
        "topK": 10,
    },
    maxResults=20,
)

reflections = response.get("memoryRecordSummaries", [])
logger.info(f"   Found {len(reflections)} relevant reflections")
if reflections:
    for reflection in reflections:
        reflection_json = json.loads(reflection["content"]["text"])
        pprint.pp(reflection_json)

## 요약

### 완료한 작업

✅ Boto3를 사용하여 reflection configuration이 포함된 episodic memory 생성

✅ 이전 debugging session으로 memory 채우기

✅ Episode 및 reflection 전용 검색 tool 구축

✅ Strands framework를 사용하여 지능형 debugging assistant 생성

✅ 전략적 지침 검색과 세부 process 검색 비교

### 핵심 내용

1. **Episodic Memory**는 시간적 맥락과 함께 전체 상호 작용 순서를 보존합니다.
2. **Reflections**는 여러 episode의 pattern과 insight를 자동으로 종합합니다.
3. **Linearization**은 LLM이 사용할 수 있도록 구조화된 data를 형식화하여 맥락을 최적화합니다.
4. **Tool 선택**이 중요합니다. Strategy에는 reflection을, 세부 단계에는 episode를 사용합니다.
5. **Boto3 Direct Access**는 Genesis Memory API operation을 완전히 제어할 수 있게 합니다.

### 이 Pattern을 사용해야 하는 경우

- 이전 ticket 해결 사례에서 학습하는 **기술 지원 system**
- 성공적인 진단 절차를 기억하는 **Troubleshooting assistant**
- 지식 전달을 위해 전문가 workflow를 포착하는 **교육 system**
- 과거 결과를 분석하여 더 나은 방식을 도출하는 **Process 개선** 시나리오

## 정리 (선택 사항)

완료 후 memory resource를 삭제하려면 주석을 해제하세요.

In [ ]:
# Boto3를 사용하여 memory resource를 삭제하려면 주석 해제
# try:
#     client.delete_memory(memoryId=memory_id, clientToken=str(uuid.uuid4()))
#     logger.info(f"✅ Successfully deleted memory: {memory_id}")
# except Exception as e:
#     logger.error(f"Error deleting memory: {e}")